In [2]:
# %pip install tensorflow
# %pip install tensorflow-cpu

### Config locale

**Détection du contrôleur VGA :**
```bash
lspci | grep -i vga
```
```text
0000:00:02.0 VGA compatible controller: Intel Corporation Raptor Lake-P [Iris Xe Graphics] (rev 04)
```

**Détails graphiques et pilotes :**
```bash
inxi -G
```
```text
Graphics:
  Device-1: Intel Raptor Lake-P [Iris Xe Graphics] driver: i915 v: kernel
  Device-2: IMC Networks USB2.0 HD UVC WebCam driver: uvcvideo type: USB
  Display: x11 server: X.Org v: 21.1.11 with: Xwayland v: 23.2.6 driver: X:
    loaded: modesetting unloaded: fbdev,vesa dri: iris gpu: i915 resolution:
    1: 1920x1080~60Hz 2: N/A
  API: EGL v: 1.5 drivers: iris,swrast platforms: gbm,x11,surfaceless,device
  API: OpenGL v: 4.6 compat-v: 4.5 vendor: intel mesa
    v: 25.2.8-0ubuntu0.24.04.4 renderer: Mesa Intel Iris Xe Graphics (RPL-P)
```

---

### Diagnostic matériel

* **Processeur graphique (GPU).** Intel Raptor Lake-P (Iris Xe Graphics), intégré au processeur.
* **Pilote actif.** `i915` (géré nativement par le noyau Linux, parfaitement installé et fonctionnel).
* **Compatibilité CUDA.** **Non compatible.** CUDA (`nvidia-smi`, `nvcc`) est une technologie propriétaire réservée uniquement aux cartes graphiques NVIDIA.

### Solutions pour exécuter vos notebooks

1. **Utiliser votre CPU en local.** Pour des tâches légères (analyse de données avec Pandas, petits modèles Scikit-learn), votre processeur Intel Raptor Lake est largement suffisant sans configuration supplémentaire.
2. **Utiliser un GPU gratuit dans le cloud.** Pour le Deep Learning (PyTorch, TensorFlow) nécessitant CUDA, exécutez vos notebooks sur **Google Colab** ou **Kaggle** (qui fournissent des GPU NVIDIA T4 ou P100 gratuitement).
3. **Accélération Intel (avancé).** Il est techniquement possible d'utiliser votre puce Intel Iris Xe pour le Machine Learning via l'extension **Intel Extension for PyTorch (IPEX)** et **oneAPI**, bien que cela soit plus complexe à configurer que CUDA.



| Modèle Ollama | RAM requise | Vitesse sur votre CPU Intel | Usage recommandé |
| :--- | :--- | :--- | :--- |
| **`qwen3:8b`** | ~5,5 - 6 Go | Moyenne (~4 à 8 mots/sec) | Tâches complexes, raisonnement, rédaction précise. Fermez vos applications lourdes avant de le lancer. |
| **`qwen3:4b`** ou **`gemma3:4b`** | ~3 - 3,5 Go | Rapide (~10 à 15 mots/sec) | **Meilleur compromis** pour un usage quotidien fluide dans votre notebook Docker. |
| **`qwen3:1.7b`** ou **`llama3.2:3b`** | ~1,5 - 2,5 Go | Très rapide (~20+ mots/sec) | Tests rapides, extraction de données, résumé de textes, prototypage. |


In [ ]:
# modèle du GPU et mémoire disponible
# !nvidia-smi
# version de CUDA
# !nvcc --version

# import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# import tensorflow as tf
# print("GPUs disponibles :", tf.config.list_physical_devices('GPU'))

In [4]:
import os
# Optionnel : réduit les messages d'avertissement de TensorFlow sur le CPU (0 = tous, 2 = erreurs uniquement)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf

# Vérification du matériel utilisé
gpus = tf.config.list_physical_devices('GPU')
cpus = tf.config.list_physical_devices('CPU')

print("GPUs disponibles :", gpus)
print("CPUs disponibles :", cpus)

I0000 00:00:1794744796.763289   10165 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794744798.170195   10165 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


GPUs disponibles : []
CPUs disponibles : [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


**Editez le service systemd**
```
sudo mkdir -p /etc/systemd/system/ollama.service.d  
echo -e '[Service]\nEnvironment="OLLAMA_HOST=0.0.0.0"' | sudo tee /etc/systemd/system/ollama.service.d/override.conf  
sudo systemctl daemon-reload  
sudo systemctl restart ollama  
```

```
ollama --version
ollama pull qwen3:8b
ollama pull qwen3:4b
ollama list
ollama show qwen3:8b
```

voir : /usr/share/ollama/.ollama/models

In [13]:
import urllib.request
import json

url = "http://172.17.0.1:11434"

try:
    # Vérifie si le serveur Ollama répond
    with urllib.request.urlopen(url, timeout=5) as response:
        print("Connexion réussie :", response.read().decode("utf-8"))
        
    # Liste les modèles téléchargés sur votre Linux Mint
    with urllib.request.urlopen(f"{url}/api/tags", timeout=5) as response:
        data = json.loads(response.read().decode("utf-8"))
        modeles = [m["name"] for m in data.get("models", [])]
        print("Modèles disponibles :", modeles)

except Exception as e:
    print("Échec de connexion :", e)

Connexion réussie : Ollama is running
Modèles disponibles : ['qwen3:8b', 'qwen3:4b']


In [ ]:
# TEST qwen3:4b

import requests
import time

url = "http://172.17.0.1:11434/api/chat"

payload = {
    "model": "qwen3:4b",
    "messages": [
        {"role": "user", "content": "Explique en 3 phrases simples comment fonctionne un réseau de neurones."}
    ],
    "stream": False  # Attend la réponse complète pour récupérer les statistiques exactes
}

debut = time.time()
response = requests.post(url, json=payload)
duree_totale = time.time() - debut

if response.status_code == 200:
    data = response.json()
    
    # Calcul de la vitesse réelle de génération (en tokens/sec)
    tokens_generes = data.get("eval_count", 0)
    duree_eval_ns = data.get("eval_duration", 1)
    vitesse = tokens_generes / (duree_eval_ns / 1e9)
    
    print("=== RÉPONSE ===")
    print(data["message"]["content"])
    print("\n=== PERFORMANCES ===")
    print(f"Temps total (chargement inclus) : {duree_totale:.2f} s")
    print(f"Tokens générés : {tokens_generes}")
    print(f"Vitesse de génération : {vitesse:.2f} tokens/sec")
else:
    print("Erreur :", response.status_code, response.text)

=== RÉPONSE ===
Un réseau de neurones est composé de plusieurs couches de nœuds connectés qui traitent des données. Chaque nœud applique des poids aux entrées et utilise une fonction d'activation pour déterminer une sortie. En ajustant ces poids grâce à des données d'entraînement, le réseau apprend à reconnaître des motifs et à faire des prédictions précises.

=== PERFORMANCES ===
Temps total (chargement inclus) : 264.72 s
Tokens générés : 1886
Vitesse de génération : 7.26 tokens/sec


In [14]:
# TEST qwen3:8b

import requests
import time

url = "http://172.17.0.1:11434/api/chat"

payload = {
    "model": "qwen3:8b",
    "messages": [
        {"role": "user", "content": "Explique en 3 phrases simples comment fonctionne un réseau de neurones."}
    ],
    "stream": False  # Attend la réponse complète pour récupérer les statistiques exactes
}

debut = time.time()
response = requests.post(url, json=payload)
duree_totale = time.time() - debut

if response.status_code == 200:
    data = response.json()
    
    # Calcul de la vitesse réelle de génération (en tokens/sec)
    tokens_generes = data.get("eval_count", 0)
    duree_eval_ns = data.get("eval_duration", 1)
    vitesse = tokens_generes / (duree_eval_ns / 1e9)
    
    print("=== RÉPONSE ===")
    print(data["message"]["content"])
    print("\n=== PERFORMANCES ===")
    print(f"Temps total (chargement inclus) : {duree_totale:.2f} s")
    print(f"Tokens générés : {tokens_generes}")
    print(f"Vitesse de génération : {vitesse:.2f} tokens/sec")
else:
    print("Erreur :", response.status_code, response.text)

=== RÉPONSE ===
Un réseau de neurones est composé de couches de neurones qui traitent les données étape par étape. Chaque neurone reçoit des informations, les combine avec des poids, et produit une sortie en utilisant une fonction d'activation. Le réseau apprend en ajustant ces poids pour minimiser les erreurs entre ses prédictions et les résultats réels.

=== PERFORMANCES ===
Temps total (chargement inclus) : 53.89 s
Tokens générés : 188
Vitesse de génération : 4.06 tokens/sec
